# S0 — Kaggle API 環境設定

> **時間**：20 分鐘  
> **學完能幹嘛**：用一行指令下載任何 Kaggle 比賽資料集到本地，開始分析

---

## 為什麼要學 Kaggle API？

手動到 Kaggle 網站下載 → 解壓 → 搬到正確資料夾，每次換資料集都要重來。

用 API 只要 **一行指令**，而且可以寫成腳本讓同學們環境統一。

**一句話口訣：手動下載一次是勤勞，自動化下載才是專業。**

---
## Step 1 — 安裝 kaggle 套件

> 需要 **Python 3.11+** 與 `pip`

In [ ]:
# 只需執行一次
# !pip install kaggle

# 如果遇到 "Command kaggle not found"，確認 Python Scripts 目錄在 $PATH 中：
# - Linux/macOS: ~/.local/bin
# - Windows: $PYTHON_HOME/Scripts

---
## Step 2 — 認證設定（三擇一）

登入 [kaggle.com](https://www.kaggle.com) → 右上角頭像 → **Settings**

### 方法 A：環境變數（推薦）

在 Settings 頁面的 **API** 區塊，點 **Generate New Token**，複製 token 後設定環境變數：

```bash
export KAGGLE_API_TOKEN=xxxxxxxxxxxxxx
```

> 可加入 `~/.bashrc` 或 `~/.zshrc` 讓每次開終端都自動載入。

### 方法 B：API Token 檔案

將 token 存入 `~/.kaggle/access_token`（純文字檔，內容就是 token 字串）。

### 方法 C：Legacy API 金鑰（舊方法）

在 Settings → **Legacy API Credentials** → **Create Legacy API Key**，會下載 `kaggle.json`，放到：
- **macOS / Linux**：`~/.kaggle/kaggle.json`
- **Windows**：`C:\Users\<你的帳號>\.kaggle\kaggle.json`

> ⚠️ 無論哪種方法，都**不要把金鑰上傳到 GitHub**！建議在 `.gitignore` 加上 `kaggle.json` 和 `access_token`。

In [ ]:
import os

# 在 Jupyter 中設定環境變數（終端的 ~/.zshrc 不會自動帶入 kernel）
# 請將下方 token 替換為你自己的 Kaggle API Token
os.environ['KAGGLE_API_TOKEN'] = 'KGAT_xxxxxxxxxxxxxxxxxxxxxxxxxxxxxx'

# 確保 kaggle CLI 在 PATH 中（macOS Homebrew Python 使用者）
user_bin = os.path.expanduser('~/Library/Python/3.12/bin')
if user_bin not in os.environ.get('PATH', ''):
    os.environ['PATH'] = user_bin + ':' + os.environ['PATH']

print('✓ 環境變數已設定')

In [ ]:
# 驗證 API 是否設定成功（任一認證方法都可以）
!kaggle competitions list --sort-by latestDeadline | head -5

---
## Kaggle CLI 指令總覽

認證設定完成後，可以用 `kaggle --help` 查看所有可用指令：

| 指令群組 | 用途 | 常用指令 |
|:---------|:-----|:---------|
| `kaggle competitions` | 比賽管理 | `list`, `download`, `submit` |
| `kaggle datasets` | 資料集搜尋/下載 | `list`, `download`, `create` |
| `kaggle kernels` | Notebooks/Scripts | `list`, `push`, `pull` |
| `kaggle models` | 模型管理 | `list`, `get`, `create` |
| `kaggle config` | CLI 設定 | `view`, `set` |

> 本課程主要使用 `kaggle competitions download` 來下載比賽資料。

from pathlib import Path
import zipfile

# 定義資料集目錄（使用相對路徑 + pathlib，跨平台通用）
BASE_DIR = Path('../datasets')

# 要下載的比賽清單
COMPETITIONS = {
    'titanic': BASE_DIR / 'titanic',
    'house-prices-advanced-regression-techniques': BASE_DIR / 'house-prices',
    'spaceship-titanic': BASE_DIR / 'spaceship-titanic',
}

for comp_name, dest_dir in COMPETITIONS.items():
    dest_dir.mkdir(parents=True, exist_ok=True)
    
    # 如果已解壓過就跳過
    csv_files = list(dest_dir.glob('*.csv'))
    if csv_files:
        print(f'✓ {comp_name} 已存在 ({len(csv_files)} 個 CSV)，跳過下載')
        continue
    
    # 下載
    print(f'⬇ 下載 {comp_name}...')
    !kaggle competitions download -c {comp_name} -p {dest_dir}
    
    # 檢查是否下載成功
    zip_files = list(dest_dir.glob('*.zip'))
    if not zip_files:
        print(f'  ✗ 下載失敗，請確認：1) API Token 已設定 2) 已同意比賽規則')
        continue
    
    # 解壓
    with zipfile.ZipFile(zip_files[0], 'r') as z:
        z.extractall(dest_dir)
    zip_files[0].unlink()  # 刪除 zip
    print(f'  ✓ 解壓完成 → {dest_dir}')

In [ ]:
from pathlib import Path
import zipfile

# 定義資料集目錄（使用相對路徑 + pathlib，跨平台通用）
BASE_DIR = Path('../datasets')

# 要下載的比賽清單
COMPETITIONS = {
    'titanic': BASE_DIR / 'titanic',
    'house-prices-advanced-regression-techniques': BASE_DIR / 'house-prices',
    'spaceship-titanic': BASE_DIR / 'spaceship-titanic',
}

for comp_name, dest_dir in COMPETITIONS.items():
    dest_dir.mkdir(parents=True, exist_ok=True)
    zip_path = dest_dir / f'{comp_name}.zip'
    
    # 如果已解壓過就跳過
    csv_files = list(dest_dir.glob('*.csv'))
    if csv_files:
        print(f'✓ {comp_name} 已存在 ({len(csv_files)} 個 CSV)，跳過下載')
        continue
    
    # 下載
    print(f'⬇ 下載 {comp_name}...')
    !kaggle competitions download -c {comp_name} -p {dest_dir}
    
    # 解壓
    zip_file = list(dest_dir.glob('*.zip'))[0]
    with zipfile.ZipFile(zip_file, 'r') as z:
        z.extractall(dest_dir)
    zip_file.unlink()  # 刪除 zip
    print(f'  ✓ 解壓完成 → {dest_dir}')

---
## Step 4 — 快速檢查每個資料集

In [ ]:
import pandas as pd

datasets = {
    'Titanic': '../datasets/titanic/train.csv',
    'House Prices': '../datasets/house-prices/train.csv',
    'Spaceship Titanic': '../datasets/spaceship-titanic/train.csv',
}

for name, path in datasets.items():
    df = pd.read_csv(path)
    print(f'\n{"="*50}')
    print(f'{name}: {df.shape[0]} rows × {df.shape[1]} columns')
    print(f'欄位: {list(df.columns[:8])}', '...' if df.shape[1] > 8 else '')
    print(f'缺值: {df.isnull().sum().sum()} 個')

---
## 環境就緒！

接下來的 S1-S5 會使用這三個資料集，學習 Kaggle 實戰分析技巧。

**下節預告 S1**：拿到一份從沒看過的資料集，前 10 分鐘該看什麼？ → **First Look Checklist**